# 99.4 Optimization Basics: Exercises

## Contents

**Tools:** CADET-Process (Python).

**Python used:** Objective functions, lists, variable bounds and constraints.

**Practises:** problem definition, starting values, local and global search.

1. Define and inspect an objective
2. Compare starting values and optimizers
3. Add a constraint

```{tip}
Run code cells from top to bottom with **Shift+Enter**. After entering, pasting or changing code, run that cell again to apply it, then rerun any later cells that use its results. Setup cells may show no output; they create objects for later cells. Fill empty answer cells as you go.
```

```{note}
**Optional material.** Chapter 99 is not part of the main sequence. Work through it after the related main topic or whenever time permits.
```

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from CADETProcess.optimization import OptimizationProblem, TrustConstr, U_NSGA3

## Exercise 1: define the objective

Minimize **$f(x)=(x-2)^3-3(x-2)+1$** within **0.2 ≤ x ≤ 4.5**.

Create `cubic_problem`, add the variable `"x"`, and register a function named `cubic`. It receives a list of values, so use `x[0]` in the formula.

In [ ]:
cubic_problem = OptimizationProblem("cubic")
cubic_problem.add_variable("x", lb=0.2, ub=4.5)

def cubic(x):
    return (x[0] - 2) ** 3 - 3 * (x[0] - 2) + 1

cubic_problem.add_objective(cubic)

The supplied plot shows the shape of the objective. Where can a search that starts at **0.5** end up? What changes when it starts at **3.8**?

In [ ]:
# Plot the objective over the allowed range.
x_values = np.linspace(0.2, 4.5, 200)
y_values = (x_values - 2) ** 3 - 3 * (x_values - 2) + 1
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(x_values, y_values)
ax.set(xlabel="x", ylabel="Objective")
ax.grid(alpha=0.2)
plt.show()

## Exercise 2: compare the searches

Run the two local searches. Compare their final `.x` and `.f` values with the plot. The optimizer runs are supplied; change `x0` if you want to try another starting value.

In [ ]:
# Run the same local optimizer from two starting values.
local_optimizer = TrustConstr()
local_optimizer.n_max_iter = 100
left_results = local_optimizer.optimize(cubic_problem, x0=[0.5])
right_results = local_optimizer.optimize(cubic_problem, x0=[3.8])
print("Start 0.5:", left_results.x, left_results.f)
print("Start 3.8:", right_results.x, right_results.f)

`U_NSGA3` searches with a population of candidates. Compare its best result with the local searches. This short run demonstrates the difference; its iteration limit does not guarantee convergence.

In [ ]:
# Run a short population-based search and show its progress.
global_optimizer = U_NSGA3()
global_optimizer.pop_size = 20
global_optimizer.n_max_iter = 15
global_optimizer.seed = 42
global_results = global_optimizer.optimize(cubic_problem)
print("Population search:", global_results.x, global_results.f)
_ = global_results.plot_convergence("objectives")
plt.show()

## Exercise 3: restrict the feasible values

Add **x ≤ 2.5** to `cubic_problem` using `add_linear_constraint()`. How does this change the best accessible point? Rerun the supplied cell after adding the constraint.

In [ ]:
cubic_problem.add_linear_constraint(["x"], lhs=[1], b=2.5)

In [ ]:
# Start inside the new feasible region and report the constrained result.
restricted_optimizer = TrustConstr()
restricted_optimizer.n_max_iter = 100
restricted_results = restricted_optimizer.optimize(cubic_problem, x0=[2.3])
print("Constrained x:", restricted_results.x)
print("Objective:", restricted_results.f)